# LoRa Predictive and Optimization Model

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from pathlib import Path
import warnings
import ee
import time
import json
import os
from dotenv import load_dotenv
from typing import Dict, List, Tuple, Optional, Union
import pickle
from scipy.interpolate import griddata
from scipy.spatial.distance import cdist
import logging
from dataclasses import dataclass, replace, asdict
from abc import ABC, abstractmethod
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

# Load environment variables
load_dotenv()
warnings.filterwarnings('ignore')


### Logging Configuration

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('lora_system.log'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Using device: {device}")
if torch.cuda.is_available():
    logger.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logger.info(f"Memory Available: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


### Constants and Physical Parameters

In [ ]:
# SNR thresholds for different spreading factors (from LoRaWAN specification)
SNR_THRESHOLD = {
    7: -7.5, 8: -10, 9: -12.5, 10: -15, 11: -17.5, 12: -20
}

# Land cover to decay constant K (derived from preprocessing analysis)
# Higher K = faster PDR recovery with good SNR margin
LAND_COVER_TO_K = {
    10: 0.25,  # Tree cover
    20: 0.28,  # Shrubland
    30: 0.32,  # Grassland
    40: 0.33,  # Cropland
    50: 0.20,  # Built-up (worst)
    60: 0.40,  # Bare/sparse vegetation
    70: 0.35,  # Snow and ice
    80: 0.45,  # Water (best)
    90: 0.22,  # Herbaceous wetland
    95: 0.20,  # Mangroves
    100: 0.30  # Moss and lichen
}

# Terrain penalty for RF propagation (0 = best, 1 = worst)
PENALTY_MAP = {
    10: 0.6,   # Tree cover - HIGH penalty
    20: 0.4,   # Shrubland - MODERATE
    30: 0.15,  # Grassland - LOW
    40: 0.25,  # Cropland - LOW-MODERATE
    50: 0.8,   # Built-up - VERY HIGH
    60: 0.2,   # Bare/sparse - LOW
    70: 0.55,  # Snow/ice - MODERATE-HIGH
    80: 0.0,   # Water - NO penalty (best)
    90: 0.35,  # Wetland - MODERATE
    95: 0.5,   # Mangroves - HIGH
    100: 0.2   # Moss/lichen - LOW
}


### Data Classes

In [ ]:
@dataclass
class LoRaParameters:
    """LoRa communication parameters with validation"""
    tx_power: float = 14.0  # dBm (2-20)
    spreading_factor: int = 7  # (7-12)
    frequency: float = 868.0  # MHz
    bandwidth: float = 125.0  # kHz
    coding_rate: int = 4  # 4/5
    
    def __post_init__(self):
        """Validate parameters after initialization"""
        if not (2 <= self.tx_power <= 30):
            raise InvalidLoRaParametersError(f"TX power {self.tx_power} must be in range [2, 30] dBm")
        if self.spreading_factor not in [7, 8, 9, 10, 11, 12]:
            raise InvalidLoRaParametersError(f"Spreading factor {self.spreading_factor} must be in [7-12]")
        if not (100 <= self.frequency <= 1000):
            raise InvalidLoRaParametersError(f"Frequency {self.frequency} must be in range [100, 1000] MHz")

@dataclass
class PathPoint:
    """Represents a point in the path with all attributes"""
    lat: float
    lon: float
    elevation: float = 0.0
    land_cover: int = 50
    terrain_penalty: float = 0.3
    rssi: float = -100.0
    snr: float = 0.0
    pdr: float = 0.5
    path_loss: float = 100.0
    distance_to_start: float = 0.0
    grid_x: int = 0
    grid_y: int = 0
    is_relay: bool = False
    hop_number: int = 0
    # Path spatial features (for 15-feature prediction)
    path_built_up_fraction: float = 0.0
    path_vegetation_fraction: float = 0.0
    path_water_fraction: float = 0.0
    path_avg_penalty: float = 0.3
    path_elevation_std: float = 0.0
    max_terrain_obstruction_m: float = 0.0
    path_dominant_land_cover: int = 50

@dataclass
class OptimizationConfig:
    """Configuration for path optimization"""
    grid_spacing_km: float = 1.5
    max_hop_distance_km: float = 5.0
    min_relay_distance_km: float = 1.0
    corridor_width_km: float = 4.0
    adaptive_grid: bool = True
    max_path_deviation: float = 0.5  # Allow 50% longer than direct path
    min_pdr_threshold: float = 0.3  # Block points with PDR < 0.3
    prefer_water: bool = True
    avoid_buildings: bool = True

@dataclass
class GEEConfig:
    """Configuration for Google Earth Engine integration"""
    batch_size: int = 50
    workers: int = 5
    retry_attempts: int = 3
    fallback_to_individual: bool = True
    cache_enabled: bool = True
    cache_file: str = 'gee_cache.pkl'
    path_spatial_samples: int = 15


### Exceptions

In [ ]:
class GEEDataUnavailableError(Exception):
    """Raised when Google Earth Engine data cannot be fetched"""
    pass

class GEEQuotaExceededError(Exception):
    """Raised when GEE API quota is exhausted"""
    pass

class InvalidCoordinatesError(Exception):
    """Raised when coordinates are out of valid range"""
    pass

class InvalidLoRaParametersError(Exception):
    """Raised when LoRa parameters are invalid"""
    pass

class NoViablePathError(Exception):
    """Raised when A* cannot find a path between start and destination"""
    pass


### Input Validation

In [ ]:
def validate_coordinates(lat: float, lon: float, name: str = "Point"):
    """Validate geographic coordinates"""
    if not (-90 <= lat <= 90):
        raise InvalidCoordinatesError(f"{name} latitude {lat} out of range [-90, 90]")
    if not (-180 <= lon <= 180):
        raise InvalidCoordinatesError(f"{name} longitude {lon} out of range [-180, 180]")

def validate_distance(start_lat: float, start_lon: float, dest_lat: float, dest_lon: float):
    """Validate that start and destination are not identical and not too far"""
    if start_lat == dest_lat and start_lon == dest_lon:
        raise InvalidCoordinatesError("Start and destination coordinates are identical")
    
    # Calculate distance
    R = 6371000  # Earth radius in meters
    phi1, phi2 = np.radians(start_lat), np.radians(dest_lat)
    dphi = np.radians(dest_lat - start_lat)
    dlambda = np.radians(dest_lon - start_lon)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    distance = R * c
    
    if distance < 500:  # Less than 500 meters
        raise InvalidCoordinatesError(f"Distance too short: {distance:.1f}m (minimum 500m)")
    
    if distance > (100 * 1000):  # More than 100 km (100,000 meters)
        logger.warning(f"Distance very large: {distance/1000:.1f}km - optimization may be slow")

def validate_lora_parameters(spreading_factor: int, tx_power: int, frequency: int):
    """Validate LoRa communication parameters"""
    # Spreading Factor
    if not isinstance(spreading_factor, int):
        raise InvalidLoRaParametersError(
            f"spreading_factor must be an integer, got {type(spreading_factor).__name__}"
        )
    if spreading_factor not in [7, 8, 9, 10, 11, 12]:
        raise InvalidLoRaParametersError(
            f"spreading_factor {spreading_factor} invalid. Must be 7, 8, 9, 10, 11, or 12. "
            f"(SF7=shortest range/fastest, SF12=longest range/slowest)"
        )
    
    # TX Power
    if not isinstance(tx_power, int):
        raise InvalidLoRaParametersError(
            f"tx_power must be a number, got {type(tx_power).__name__}"
        )
    if not (2 <= tx_power <= 30):
        raise InvalidLoRaParametersError(
            f"tx_power {tx_power} dBm out of range [2, 30]. "
            f"Typical values: 14 dBm (standard), 20 dBm (high power)"
        )
    if tx_power > 20:
        logger.warning(
            f"TX power {tx_power} dBm is very high. "
            f"Ensure your hardware supports this. Typical max: 20 dBm"
        )
    
    # Frequency
    if not isinstance(frequency, int):
        raise InvalidLoRaParametersError(
            f"frequency must be a number, got {type(frequency).__name__}"
        )
    if not (200 <= frequency <= 1000):
        raise InvalidLoRaParametersError(
            f"frequency {frequency} MHz out of range [200, 1000]. "
            f"Common bands: EU=868, US=915, AS=923, IN=865"
        )
    
    # Frequency band warnings
    if 863 <= frequency <= 870:
        logger.info("Using EU863-870 band (Europe)")
    elif 902 <= frequency <= 928:
        logger.info("Using US902-928 band (North America)")
    elif 915 <= frequency <= 928:
        logger.info("Using AS923 band (Asia)")
    else:
        logger.warning(
            f"Frequency {frequency} MHz is unusual. "
            f"Standard bands: EU=868, US=915, AS=923"
        )

def validate_grid_parameters(grid_spacing_km: float, corridor_width_km: float, 
                            adaptive_grid: bool):
    """Validate grid configuration parameters"""
    # Grid Spacing
    if not isinstance(grid_spacing_km, (int, float)):
        raise ValueError(
            f"grid_spacing_km must be a number, got {type(grid_spacing_km).__name__}"
        )
    if not (0.2 <= grid_spacing_km <= 10):
        raise ValueError(
            f"grid_spacing_km {grid_spacing_km} out of range [0.2, 10]. "
            f"Recommended: 1.0-2.0 km for best results"
        )
    if grid_spacing_km < 0.5:
        logger.warning(
            f"grid_spacing_km {grid_spacing_km} is very small. "
            f"This will create a very dense grid (slow computation)"
        )
    if grid_spacing_km > 5:
        logger.warning(
            f"grid_spacing_km {grid_spacing_km} is very large. "
            f"This may miss optimal paths. Recommended: 1.0-2.0 km"
        )
    
    # Corridor Width
    if not isinstance(corridor_width_km, (int, float)):
        raise ValueError(
            f"corridor_width_km must be a number, got {type(corridor_width_km).__name__}"
        )
    if not (0.5 <= corridor_width_km <= 20):
        raise ValueError(
            f"corridor_width_km {corridor_width_km} out of range [0.5, 20]. "
            f"Recommended: 3.0-6.0 km"
        )
    if corridor_width_km < 2:
        logger.warning(
            f"corridor_width_km {corridor_width_km} is narrow. "
            f"Path may not find good alternatives around obstacles"
        )
    
    # Adaptive Grid
    if not isinstance(adaptive_grid, bool):
        raise ValueError(
            f"adaptive_grid must be True or False, got {type(adaptive_grid).__name__}"
        )

def validate_gee_parameters(gee_workers: int):
    """Validate Google Earth Engine parameters"""
    if not isinstance(gee_workers, int):
        raise ValueError(
            f"gee_workers must be an integer, got {type(gee_workers).__name__}"
        )
    if not (1 <= gee_workers <= 20):
        raise ValueError(
            f"gee_workers {gee_workers} out of range [1, 20]. "
            f"Recommended: 5-10 for best speed/stability"
        )
    if gee_workers > 10:
        logger.warning(
            f"gee_workers {gee_workers} is very high. "
            f"May hit API rate limits. Recommended: 5-10"
        )

def validate_optimization_parameters(max_path_deviation: float, min_pdr_threshold: float,
                                    prefer_water: bool, avoid_buildings: bool,
                                    direct_path_threshold_km: float):
    """Validate optimization preference parameters"""
    # Max Path Deviation
    if not isinstance(max_path_deviation, (int, float)):
        raise ValueError(
            f"max_path_deviation must be a number, got {type(max_path_deviation).__name__}"
        )
    if not (0.0 <= max_path_deviation <= 3.0):
        raise ValueError(
            f"max_path_deviation {max_path_deviation} out of range [0.0, 3.0]. "
            f"0.5 = allow 50% longer path, 1.0 = allow 100% longer (double length). "
            f"Recommended: 0.3-1.0"
        )
    if max_path_deviation < 0.1:
        logger.warning(
            f"max_path_deviation {max_path_deviation} is very strict. "
            f"Path will be nearly straight. May fail to find route."
        )
    if max_path_deviation > 1.5:
        logger.warning(
            f"max_path_deviation {max_path_deviation} is very loose. "
            f"Path may zigzag excessively. Recommended: 0.3-1.0"
        )
    
    # Min PDR Threshold
    if not isinstance(min_pdr_threshold, (int, float)):
        raise ValueError(
            f"min_pdr_threshold must be a number, got {type(min_pdr_threshold).__name__}"
        )
    if not (0.0 <= min_pdr_threshold <= 1.0):
        raise ValueError(
            f"min_pdr_threshold {min_pdr_threshold} out of range [0.0, 1.0]. "
            f"0.3 = 30% minimum PDR, 0.5 = 50% minimum. "
            f"Recommended: 0.2-0.5"
        )
    if min_pdr_threshold < 0.1:
        logger.warning(
            f"min_pdr_threshold {min_pdr_threshold} is very low. "
            f"Path may use poor quality links. Recommended: 0.2-0.5"
        )
    if min_pdr_threshold > 0.6:
        logger.warning(
            f"min_pdr_threshold {min_pdr_threshold} is very high. "
            f"May fail to find route. Recommended: 0.2-0.5"
        )
    
    # Prefer Water
    if not isinstance(prefer_water, bool):
        raise ValueError(
            f"prefer_water must be True or False, got {type(prefer_water).__name__}"
        )
    
    # Avoid Buildings
    if not isinstance(avoid_buildings, bool):
        raise ValueError(
            f"avoid_buildings must be True or False, got {type(avoid_buildings).__name__}"
        )
    
    # Direct Path Threshold
    if not isinstance(direct_path_threshold_km, (int, float)):
        raise ValueError(
            f"direct_path_threshold_km must be a number, got {type(direct_path_threshold_km).__name__}"
        )
    if not (0.1 <= direct_path_threshold_km <= 10.0):
        raise ValueError(
            f"direct_path_threshold_km {direct_path_threshold_km} out of range [0.1, 10.0]. "
            f"1.0 = use direct path for distances < 1 km. "
            f"Recommended: 0.5-2.0"
        )
    if direct_path_threshold_km > 5.0:
        logger.warning(
            f"direct_path_threshold_km {direct_path_threshold_km} is very large. "
            f"System will attempt direct links over long distances. "
            f"This may result in poor quality. Recommended: 0.5-2.0"
        )


### Lora Physics Engineering

In [ ]:
class LoRaPhysicsEngine:
    """
    Pure physics-based LoRa calculations (NOT machine learning)
    
    This class handles all physics formulas for LoRa communication:
    - PDR calculation from SNR (exponential decay model)
    - Link budget calculations
    - Sensitivity thresholds
    
    These are NOT predicted by ML models, but calculated using established
    radio propagation formulas and LoRaWAN specifications.
    """
    
    def __init__(self):
        self.snr_thresholds = SNR_THRESHOLD
        self.land_cover_k = LAND_COVER_TO_K
    
    def calculate_pdr(self, snr: float, spreading_factor: int, land_cover: int) -> float:
        """
        Calculate Packet Delivery Rate from SNR using exponential decay model
        
        Formula: PDR = 1 - exp(-k * margin)
        Where margin = SNR - SNR_threshold
        
        Args:
            snr: Signal-to-Noise Ratio (dB)
            spreading_factor: LoRa spreading factor (7-12)
            land_cover: ESA WorldCover land cover code
        
        Returns:
            PDR value between 0.0 and 1.0
        """
        snr_threshold = self.snr_thresholds.get(spreading_factor, -7.5)
        margin = snr - snr_threshold
        
        # No signal if below threshold
        if margin <= 0:
            return 0.0
        
        # Get decay constant based on land cover
        k = self.land_cover_k.get(land_cover, 0.3)
        
        # Exponential recovery formula
        pdr = 1 - np.exp(-k * margin)
        
        # Clamp to [0, 1]
        return max(0.0, min(1.0, pdr))
    
    def get_sensitivity(self, spreading_factor: int) -> float:
        """Get receiver sensitivity for given SF"""
        sensitivity_map = {
            7: -123, 8: -126, 9: -129, 10: -132, 11: -134, 12: -137
        }
        return sensitivity_map.get(spreading_factor, -123)


### Google Earth Engine Integration

In [ ]:
class RateLimiter:
    """Simple rate limiter for API calls"""
    def __init__(self, calls_per_second=10):
        self.calls_per_second = calls_per_second
        self.last_call = 0
        
    def __enter__(self):
        elapsed = time.time() - self.last_call
        if elapsed < 1.0 / self.calls_per_second:
            time.sleep((1.0 / self.calls_per_second) - elapsed)
        self.last_call = time.time()
        
    def __exit__(self, exc_type, exc_val, exc_tb):
        pass


class BatchGEEIntegration:
    """
    Robust batch spatial data fetching from Google Earth Engine with parallel workers
    
    STRATEGY:
    1. Try batch request (50 points) - FAST but may fail
    2. If batch fails → Split into smaller chunks (10 points)
    3. If chunks fail → Individual calls (slowest but most reliable)
    
    Features:
    - Configurable parallel workers (default 5)
    - Automatic retry logic
    - Disk caching for reuse
    - Progress tracking with ETA
    """
    
    def __init__(self, config: GEEConfig):
        self.config = config
        self.initialized = False
        self.cache = {}
        self.rate_limiter = RateLimiter(calls_per_second=10)
        self.physics_engine = LoRaPhysicsEngine()
        
        # Load cache from disk if exists
        if config.cache_enabled and os.path.exists(config.cache_file):
            try:
                with open(config.cache_file, 'rb') as f:
                    self.cache = pickle.load(f)
                logger.info(f"Loaded {len(self.cache)} cached GEE results from {config.cache_file}")
            except Exception as e:
                logger.warning(f"Could not load cache: {e}")
        
        # Initialize Google Earth Engine
        self._initialize_gee()
    
    def _initialize_gee(self):
        """Initialize GEE with error handling"""
        try:
            project_id = os.getenv('GEE_PROJECT_ID')
            if project_id:
                ee.Initialize(project=project_id)
                logger.info(f"Google Earth Engine initialized with project ID")
            else:
                ee.Initialize()
                logger.info("Google Earth Engine initialized")
            
            # Test with simple request
            test_point = ee.Geometry.Point([26, 26])
            test_result = ee.Image('USGS/SRTMGL1_003').sample(test_point, scale=30).getInfo()
            self.initialized = True
            logger.info("GEE test successful - ready for batch operations")
            
        except Exception as e:
            logger.error(f"Google Earth Engine initialization failed: {e}")
            logger.error("Please check GEE credentials and authentication")
            raise GEEDataUnavailableError(f"Cannot initialize GEE: {e}")
    
    def _get_cache_key(self, lat: float, lon: float, data_type: str) -> str:
        """Generate cache key for coordinate and data type"""
        return f"{data_type}_{lat:.6f}_{lon:.6f}"
    
    def get_elevation(self, lat: float, lon: float) -> float:
        """Fetch elevation from SRTM (30m resolution)"""
        cache_key = self._get_cache_key(lat, lon, 'elevation')
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        try:
            with self.rate_limiter:
                point = ee.Geometry.Point([lon, lat])
                srtm = ee.Image('USGS/SRTMGL1_003')
                elevation_dict = srtm.reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=point,
                    scale=30,
                    maxPixels=1
                ).getInfo()
                
                elevation = elevation_dict.get('elevation')
                if elevation is not None:
                    elevation = float(elevation)
                    self.cache[cache_key] = elevation
                    return elevation
                else:
                    raise GEEDataUnavailableError(f"No elevation data at ({lat}, {lon})")
                    
        except Exception as e:
            logger.error(f"Failed to get elevation for ({lat}, {lon}): {e}")
            raise GEEDataUnavailableError(f"Elevation fetch failed: {e}")
    
    def get_land_cover(self, lat: float, lon: float) -> Tuple[int, float]:
        """Fetch land cover from ESA WorldCover (10m resolution)"""
        cache_key = self._get_cache_key(lat, lon, 'landcover')
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        try:
            with self.rate_limiter:
                point = ee.Geometry.Point([lon, lat])
                worldcover = ee.ImageCollection('ESA/WorldCover/v200').first()
                lc_dict = worldcover.reduceRegion(
                    reducer=ee.Reducer.first(),
                    geometry=point,
                    scale=10,
                    maxPixels=1
                ).getInfo()
                
                land_cover = lc_dict.get('Map')
                if land_cover is not None:
                    land_cover_code = int(land_cover)
                    terrain_penalty = PENALTY_MAP.get(land_cover_code, 0.5)
                    result = (land_cover_code, terrain_penalty)
                    self.cache[cache_key] = result
                    return result
                else:
                    # Default to water if no data
                    return 80, 0.0
                    
        except Exception as e:
            logger.error(f"Failed to get land cover for ({lat}, {lon}): {e}")
            raise GEEDataUnavailableError(f"Land cover fetch failed: {e}")
    
    def get_spatial_features(self, lat: float, lon: float) -> Dict:
        """Fetch all spatial features for a single location"""
        cache_key = self._get_cache_key(lat, lon, 'spatial')
        if cache_key in self.cache:
            return self.cache[cache_key]
        
        elevation = self.get_elevation(lat, lon)
        land_cover, terrain_penalty = self.get_land_cover(lat, lon)
        
        result = {
            'elevation': elevation,
            'land_cover': land_cover,
            'terrain_penalty': terrain_penalty
        }
        
        self.cache[cache_key] = result
        return result
    
    def get_path_spatial_features(self, lat1: float, lon1: float, 
                                  lat2: float, lon2: float) -> Dict:
        """
        Compute 7 path-based spatial features between two points
        
        Samples points along the line and calculates:
        - Fraction of built-up areas
        - Fraction of vegetation
        - Fraction of water
        - Average terrain penalty
        - Elevation standard deviation
        - Maximum terrain obstruction
        - Dominant land cover
        """
        num_samples = self.config.path_spatial_samples
        
        # Generate intermediate points
        lats = np.linspace(lat1, lat2, num_samples)
        lons = np.linspace(lon1, lon2, num_samples)
        
        built_up = veg = water = 0
        penalties = []
        elevations = []
        land_covers = []
        
        for lat, lon in zip(lats, lons):
            try:
                lc, penalty = self.get_land_cover(lat, lon)
                elev = self.get_elevation(lat, lon)
                
                # Count land cover types
                if lc == 50:
                    built_up += 1
                elif lc in {10, 20, 90, 95}:
                    veg += 1
                elif lc == 80:
                    water += 1
                
                penalties.append(penalty)
                elevations.append(elev)
                land_covers.append(lc)
                
            except GEEDataUnavailableError:
                continue
        
        total = len(elevations) if elevations else 1
        
        return {
            'path_built_up_fraction': built_up / total,
            'path_vegetation_fraction': veg / total,
            'path_water_fraction': water / total,
            'path_avg_penalty': np.mean(penalties) if penalties else 0.3,
            'path_elevation_std': np.std(elevations) if len(elevations) > 1 else 0.0,
            'max_terrain_obstruction_m': float(np.max(elevations) - np.min(elevations)) if elevations else 0.0,
            'path_dominant_land_cover': int(np.median(land_covers)) if land_covers else 50
        }
    
    def batch_fetch_spatial_features(self, coordinates: List[Tuple[float, float]]) -> List[Dict]:
        """
        Fetch spatial features for multiple locations using parallel workers
        
        Args:
            coordinates: List of (lat, lon) tuples
        
        Returns:
            List of dictionaries with spatial features
        """
        total = len(coordinates)
        logger.info(f"Fetching spatial features for {total} locations with {self.config.workers} workers...")
        
        results = [None] * total
        
        # Use ThreadPoolExecutor for parallel fetching
        with ThreadPoolExecutor(max_workers=self.config.workers) as executor:
            # Submit all tasks
            future_to_idx = {
                executor.submit(self.get_spatial_features, lat, lon): idx
                for idx, (lat, lon) in enumerate(coordinates)
            }
            
            # Progress bar
            with tqdm(total=total, desc="GEE Batch Fetch", unit="points") as pbar:
                for future in as_completed(future_to_idx):
                    idx = future_to_idx[future]
                    try:
                        result = future.result()
                        result['latitude'] = coordinates[idx][0]
                        result['longitude'] = coordinates[idx][1]
                        results[idx] = result
                    except Exception as e:
                        logger.warning(f"Failed to fetch data for point {idx}: {e}")
                        # Use default values
                        results[idx] = {
                            'latitude': coordinates[idx][0],
                            'longitude': coordinates[idx][1],
                            'elevation': 0.0,
                            'land_cover': 50,
                            'terrain_penalty': 0.5
                        }
                    pbar.update(1)
        
        # Save cache to disk
        if self.config.cache_enabled:
            try:
                with open(self.config.cache_file, 'wb') as f:
                    pickle.dump(self.cache, f)
                logger.info(f"Saved {len(self.cache)} GEE results to cache")
            except Exception as e:
                logger.warning(f"Could not save cache: {e}")
        
        logger.info(f"Batch fetch completed: {total} points processed")
        return results


### Data Loading and Preprocessing

In [ ]:
class UnifiedFeatureBuilder:
    """
    Builds consistent 15-feature vectors for all predictions
    
    Features:
    1. elevation
    2. land_cover
    3. terrain_penalty
    4. distance_to_start
    5. spreading_factor
    6. frequency
    7. tx_power
    8. elevation_normalized
    9. path_built_up_fraction
    10. path_vegetation_fraction
    11. path_water_fraction
    12. path_avg_penalty
    13. path_elevation_std
    14. max_terrain_obstruction_m
    15. path_dominant_land_cover
    """
    
    @staticmethod
    def build_feature_vector(point: PathPoint, lora_params: LoRaParameters) -> np.ndarray:
        """
        Build 15-feature vector for ML prediction
        
        Args:
            point: PathPoint with all spatial and path features populated
            lora_params: LoRa communication parameters
        
        Returns:
            numpy array of shape (1, 15)
        """
        features = np.array([[
            point.elevation,                        # 1
            point.land_cover,                       # 2
            point.terrain_penalty,                  # 3
            point.distance_to_start,                # 4
            lora_params.spreading_factor,           # 5
            lora_params.frequency,                  # 6
            lora_params.tx_power,                   # 7
            point.elevation / 1000.0,               # 8 - normalized
            point.path_built_up_fraction,           # 9
            point.path_vegetation_fraction,         # 10
            point.path_water_fraction,              # 11
            point.path_avg_penalty,                 # 12
            point.path_elevation_std,               # 13
            point.max_terrain_obstruction_m,        # 14
            point.path_dominant_land_cover          # 15
        ]])
        
        return features
    
    @staticmethod
    def validate_feature_count(features: np.ndarray):
        """Validate that feature vector has correct shape"""
        if features.shape[1] != 15:
            raise ValueError(f"Expected 15 features, got {features.shape[1]}")

class LoRaDataPreprocessor:
    """Data loading and preprocessing with flexible format handling"""
    
    def __init__(self, gee_integration: Optional[BatchGEEIntegration] = None):
        self.scaler = StandardScaler()
        self.gee = gee_integration

    def load_dataset1(self, filepath):
        """Load dataset with format: latitude,longitude,elevation,land_cover,etc."""
        try:
            for sep in [',', '|', '\t']:
                try:
                    df = pd.read_csv(filepath, sep=sep)
                    if len(df.columns) > 5:
                        break
                except:
                    continue
            else:
                raise ValueError("Could not determine file format")
            
            column_mapping = {
                'latitude': ['latitude', 'lat'],
                'longitude': ['longitude', 'lon'],
                'elevation': ['elevation', 'altitude', 'elev'],
                'land_cover': ['land_cover', 'land_cover_code', 'landcover'],
                'terrain_penalty': ['terrain_penalty', 'terrain'],
                'RSSI': ['RSSI', 'rssi'],
                'SNR': ['SNR', 'snr'],
                'observed_path_loss': ['observed_path_loss', 'path_loss', 'loss'],
                'spreading_factor': ['spreading_factor', 'sf'],
                'frequency': ['frequency', 'freq'],
                'tx_power': ['tx_power', 'power'],
                'distance_to_start': ['distance_to_start'],
                'path_built_up_fraction': ['path_built_up_fraction'],
                'path_vegetation_fraction': ['path_vegetation_fraction'],
                'path_water_fraction': ['path_water_fraction'],
                'path_avg_penalty': ['path_avg_penalty'],
                'path_elevation_std': ['path_elevation_std'],
                'max_terrain_obstruction_m': ['max_terrain_obstruction_m'],
                'path_dominant_land_cover': ['path_dominant_land_cover']
            }
            
            df_processed = pd.DataFrame()
            for std_col, possible_cols in column_mapping.items():
                for col in possible_cols:
                    if col in df.columns:
                        df_processed[std_col] = df[col]
                        break
                else:
                    # Set defaults for missing columns
                    if std_col == 'spreading_factor':
                        df_processed[std_col] = 7
                    elif std_col == 'frequency':
                        df_processed[std_col] = 868
                    elif std_col == 'tx_power':
                        df_processed[std_col] = 14
                    elif std_col in ['terrain_penalty', 'path_avg_penalty']:
                        df_processed[std_col] = 0.3
                    elif std_col in ['path_built_up_fraction', 'path_vegetation_fraction', 
                                   'path_water_fraction', 'path_elevation_std', 
                                   'max_terrain_obstruction_m']:
                        df_processed[std_col] = 0.0
                    elif std_col == 'path_dominant_land_cover':
                        df_processed[std_col] = 50
                    else:
                        df_processed[std_col] = 0
            
            return df_processed.dropna(subset=['RSSI', 'SNR'])
            
        except Exception as e:
            logger.error(f"Error loading dataset: {e}")
            return pd.DataFrame()

    def load_dataset2(self, filepath):
        """Load dataset with format: device_id,gateway_id,latitude,longitude,etc."""
        return self.load_dataset1(filepath)  # Same logic

    def merge_datasets(self, df1, df2):
        """Merge and clean datasets"""
        if df1.empty and df2.empty:
            raise ValueError("Both datasets are empty!")
        if df1.empty:
            df_combined = df2.copy()
        elif df2.empty:
            df_combined = df1.copy()
        else:
            df_combined = pd.concat([df1, df2], ignore_index=True)
        
        df_combined = df_combined.dropna(subset=['RSSI', 'SNR'])
        df_combined['elevation_normalized'] = df_combined['elevation'] / 1000
        
        if 'frequency' not in df_combined.columns:
            df_combined['frequency'] = 868
        if 'tx_power' not in df_combined.columns:
            df_combined['tx_power'] = 14
        
        logger.info(f"Combined dataset shape: {df_combined.shape}")
        return df_combined

    def prepare_features(self, df, target_cols=['RSSI', 'SNR', 'observed_path_loss']):
        """Prepare 15-feature dataset for training"""
        feature_cols = [
            'elevation', 'land_cover', 'terrain_penalty',
            'distance_to_start',
            'spreading_factor', 'frequency', 'tx_power',
            'elevation_normalized',
            'path_built_up_fraction',
            'path_vegetation_fraction',
            'path_water_fraction',
            'path_avg_penalty',
            'path_elevation_std',
            'max_terrain_obstruction_m',
            'path_dominant_land_cover'
        ]
        
        missing_cols = [col for col in feature_cols if col not in df.columns]
        if missing_cols:
            raise ValueError(f"Missing feature columns: {missing_cols}")
        
        X = df[feature_cols].values
        y = df[target_cols].values
        
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=42
        )
        
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        
        logger.info(f"Training samples: {len(X_train)}")
        logger.info(f"Test samples: {len(X_test)}")
        logger.info(f"Features: {len(feature_cols)} (15-feature model)")
        
        return X_train_scaled, X_test_scaled, y_train, y_test, feature_cols


### Pytroch Neural Network Model

In [ ]:
class LoRaDataset(Dataset):
    """PyTorch dataset for LoRa data"""
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
        
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

class LoRaNeuralNetwork(nn.Module):
    """Neural network for RSSI, SNR, path_loss prediction"""
    def __init__(self, input_size, output_size, config=None):
        super(LoRaNeuralNetwork, self).__init__()
        
        default_config = {
            'hidden_sizes': [256, 128, 64, 32],
            'dropout_rate': 0.3,
            'activation': 'relu',
            'batch_norm': True,
            'residual_connections': True
        }
        if config:
            default_config.update(config)
        self.config = default_config
        
        layers = []
        prev_size = input_size
        
        for i, hidden_size in enumerate(self.config['hidden_sizes']):
            layers.append(nn.Linear(prev_size, hidden_size))
            
            if self.config['batch_norm']:
                layers.append(nn.BatchNorm1d(hidden_size))
            
            if self.config['activation'] == 'relu':
                layers.append(nn.ReLU())
            elif self.config['activation'] == 'leaky_relu':
                layers.append(nn.LeakyReLU(0.2))
            elif self.config['activation'] == 'elu':
                layers.append(nn.ELU())
            
            if self.config['dropout_rate'] > 0:
                layers.append(nn.Dropout(self.config['dropout_rate']))
            
            prev_size = hidden_size
        
        layers.append(nn.Linear(prev_size, output_size))
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

    def predict(self, X):
        """Make predictions with the model"""
        self.eval()
        with torch.no_grad():
            if isinstance(X, np.ndarray):
                X = torch.FloatTensor(X)
            X = X.to(next(self.parameters()).device)
            predictions = self.forward(X).cpu().numpy()
        return predictions

class NeuralNetworkTrainer:
    """Neural network trainer with early stopping"""
    def __init__(self, input_size, output_size, device, config=None):
        self.device = device
        self.config = config or {}
        
        model_config = self.config.get('model', {})
        self.model = LoRaNeuralNetwork(input_size, output_size, model_config).to(device)
        
        self.criterion = nn.MSELoss()
        self.optimizer = optim.Adam(
            self.model.parameters(), 
            lr=self.config.get('learning_rate', 0.001),
            weight_decay=self.config.get('weight_decay', 1e-5)
        )
        
        scheduler_type = self.config.get('scheduler', 'reduce_on_plateau')
        if scheduler_type == 'reduce_on_plateau':
            self.scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                self.optimizer, mode='min', patience=10, factor=0.5, verbose=True
            )
        elif scheduler_type == 'cosine':
            self.scheduler = optim.lr_scheduler.CosineAnnealingLR(
                self.optimizer, T_max=self.config.get('epochs', 100)
            )
        
        self.early_stopping_patience = self.config.get('early_stopping_patience', 20)
        self.early_stopping_counter = 0
        self.best_val_loss = float('inf')
        self.train_losses = []
        self.val_losses = []

    def train(self, train_loader, val_loader, epochs=None):
        """Train the neural network"""
        epochs = epochs or self.config.get('epochs', 100)
        logger.info(f"\nTraining Neural Network on {self.device}...")
        
        for epoch in range(epochs):
            # Training phase
            self.model.train()
            train_loss = 0
            train_steps = 0
            
            for X_batch, y_batch in train_loader:
                X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                
                self.optimizer.zero_grad()
                outputs = self.model(X_batch)
                loss = self.criterion(outputs, y_batch)
                loss.backward()
                
                if self.config.get('gradient_clip', 0) > 0:
                    nn.utils.clip_grad_norm_(self.model.parameters(), self.config['gradient_clip'])
                
                self.optimizer.step()
                train_loss += loss.item()
                train_steps += 1
            
            train_loss /= train_steps
            self.train_losses.append(train_loss)
            
            # Validation phase
            self.model.eval()
            val_loss = 0
            val_steps = 0
            
            with torch.no_grad():
                for X_batch, y_batch in val_loader:
                    X_batch, y_batch = X_batch.to(self.device), y_batch.to(self.device)
                    outputs = self.model(X_batch)
                    loss = self.criterion(outputs, y_batch)
                    val_loss += loss.item()
                    val_steps += 1
            
            val_loss /= val_steps
            self.val_losses.append(val_loss)
            
            # Learning rate scheduling
            if isinstance(self.scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                self.scheduler.step(val_loss)
            else:
                self.scheduler.step()
            
            # Early stopping
            if val_loss < self.best_val_loss:
                self.best_val_loss = val_loss
                self.early_stopping_counter = 0
                torch.save(self.model.state_dict(), 'best_model.pth')
            else:
                self.early_stopping_counter += 1
                if self.early_stopping_counter >= self.early_stopping_patience:
                    logger.info(f"Early stopping at epoch {epoch+1}")
                    self.model.load_state_dict(torch.load('best_model.pth'))
                    break
            
            if (epoch + 1) % 10 == 0:
                logger.info(f"Epoch [{epoch+1}/{epochs}] - Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
        
        logger.info("Neural Network training completed!")

    def predict(self, X):
        """Make predictions"""
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X).to(self.device)
            predictions = self.model(X_tensor).cpu().numpy()
        return predictions


### Random Forest Model

In [ ]:
class RandomForestModel:
    """Random Forest model for RSSI, SNR, path_loss prediction"""
    def __init__(self, n_estimators=100):
        self.models = {
            'RSSI': RandomForestRegressor(n_estimators=n_estimators, random_state=42, n_jobs=-1),
            'SNR': RandomForestRegressor(n_estimators=n_estimators, random_state=42, n_jobs=-1),
            'path_loss': RandomForestRegressor(n_estimators=n_estimators, random_state=42, n_jobs=-1)
        }

    def train(self, X_train, y_train):
        """Train all models"""
        logger.info("\nTraining Random Forest models...")
        for i, (name, model) in enumerate(self.models.items()):
            logger.info(f"Training {name} model...")
            model.fit(X_train, y_train[:, i])
        logger.info("Random Forest training completed!")

    def predict(self, X):
        """Make predictions"""
        predictions = np.zeros((X.shape[0], len(self.models)))
        for i, model in enumerate(self.models.values()):
            predictions[:, i] = model.predict(X)
        return predictions

    def get_feature_importance(self, feature_names):
        """Get feature importance"""
        importance_dict = {}
        for name, model in self.models.items():
            importance_dict[name] = dict(zip(feature_names, model.feature_importances_))
        return importance_dict


### XGBoost Model

In [ ]:
class XGBoostModel:
    """XGBoost model for RSSI, SNR, path_loss prediction"""
    def __init__(self):
        self.models = {
            'RSSI': xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, 
                                    random_state=42, n_jobs=-1),
            'SNR': xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, 
                                   random_state=42, n_jobs=-1),
            'path_loss': xgb.XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, 
                                         random_state=42, n_jobs=-1)
        }

    def train(self, X_train, y_train):
        """Train all models"""
        logger.info("\nTraining XGBoost models...")
        for i, (name, model) in enumerate(self.models.items()):
            logger.info(f"Training {name} model...")
            model.fit(X_train, y_train[:, i])
        logger.info("XGBoost training completed!")

    def predict(self, X):
        """Make predictions"""
        predictions = np.zeros((X.shape[0], len(self.models)))
        for i, model in enumerate(self.models.values()):
            predictions[:, i] = model.predict(X)
        return predictions
